In [0]:
# %pip install databricks-feature-engineering

In [0]:
import sys
for module in list(sys.modules.keys()):
    if "amfs_tp" in module:
        del sys.modules[module]

dbutils.library.restartPython()

In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# OPTION 1: Path Setup (until we build the wheel)
import sys, os
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
config_dir = os.path.abspath(os.path.join(current_dir, '..', 'configs'))
if src_path not in sys.path: sys.path.append(src_path)

# COMMAND ----------
import logging
from amfs_tp.pipeline.handler_job import HandlerJob
from amfs_tp.pipeline.feature_job import FeatureJob
from amfs_tp.pipeline.inference_job import InferenceJob

logging.basicConfig(level=logging.INFO)
snapshot = "202505"
catalog = "amfs_tm"

# COMMAND ----------
# PHASE 1: Data Cleaning (Raw -> Clean)
# print("🚀 Step 1: Running Handler Job...")
# handler = HandlerJob(spark, catalog=catalog)
# handler.run_cleaning(snapshot)

# # COMMAND ----------
# # PHASE 2: Feature Engineering (Clean -> Features)
# print("\n🚀 Step 2: Running Feature Engineering Job...")
# feat_config_path = f"{config_dir}/feature_config.yaml"
# feature_job = FeatureJob(
#     spark, 
#     catalog=catalog, 
#     clean_schema="clean", 
#     features_schema="features",
#     config_path=feat_config_path
# )
# feature_job.run(snapshot)

# # COMMAND ----------
# # PHASE 3: Verification
# print("\n--- Final Features Created in amfs_tm.features ---")
# display(spark.sql(f"SHOW TABLES IN {catalog}.features"))

# # COMMAND ----------
# # PHASE 3: Filtering, Matrix Generation, and Inference
# print("\n🚀 Step 3: Starting Inference Pipeline...")

# # Paths to the configs you uploaded
# filter_cfg_path = f"{config_dir}/filters_config.yaml"
# matrix_cfg_path = f"{config_dir}/matrix_config.yaml"
# impute_cfg_path = f"{config_dir}/impute_config.yaml" # New YAML path

# # Initialize the Scorer
# inference_job = InferenceJob(
#     spark, 
#     catalog=catalog, 
#     filter_cfg_path=filter_cfg_path, 
#     matrix_cfg_path=matrix_cfg_path,
#     impute_cfg_path=impute_cfg_path # Pass it here
# )

# # Run the complete flow: 
# # Filtering (Audit) -> Matrix (Merge/Impute) -> Scoring
# scored_leads_table = inference_job.run_inference(snapshot)

from pyspark.sql import functions as F

# COMMAND ----------
# PHASE 4: Post-Inference Verification
print(f"\n✅ Inference Complete. Leads table: {scored_leads_table}")

# 1. View Top Leads
print("\nTop 10 High-Propensity Customers:")
display(spark.table(scored_leads_table).orderBy(F.desc("propensity_score")).limit(10))

# 2. View Filter Audit (Rules counts)
print("\nFiltering Audit Trail (CIF Counts per Rule):")
# Table generated during FilterEngine.apply_filters logic
display(spark.table(f"{catalog}.audit.filter_stats").filter(f"snapshot_date = '{snapshot}'"))

# 3. Verify Final Matrix
print("\nImputed/Dummified Matrix Sample:")
display(spark.table(f"{catalog}.models.inference_matrix_{snapshot}").limit(5))

In [0]:
%sql
SELECT count(*) 
FROM amfs_tm.features.bank_ph_features 
WHERE snapshot_date = '202505'